<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=344341119" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 13 SESSION: FULL REBUILD + FEATURE EXTRACTION (once) + MMD UNDER 3 GAMMA CONVENTIONS =====
# Produces 17 self-explanatory files. Extraction happens ONCE; all three gamma conventions
# (global, per-architecture, fixed) are computed from the same cached features, no re-extraction.

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import gc
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import load_model, Model
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'
SS_DIR        = '/kaggle/input/datasets/asivakumarnair/drstage11singlesource/'
WEIGHTS_DIR   = '/kaggle/input/datasets/asivakumarnair/drbestmodels/'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED, EYEPACS_TARGET = 42, 3662
FEATURE_SAMPLE_N = 500  # cap per source, same convention as chest's Stage 13

# ---------- DATA REBUILD, all three sources ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['patient_id'] = None

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError:
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return pick(p_tr), pick(p_va), pick(p_te)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")
e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")
m_tr, m_va, m_te = split_messidor_mixed(messidor)
test_sets_full = {'aptos': a_te, 'eyepacs': e_te, 'messidor': m_te}

test_sets = {}
for src, df in test_sets_full.items():
    test_sets[src] = df.sample(min(FEATURE_SAMPLE_N, len(df)), random_state=SEED).reset_index(drop=True)
    print(f"{src}: using {len(test_sets[src])} of {len(df)} test images for feature extraction")

# ---------- MODEL PATHS ----------
ARCH_SPECS   = {'custom': None, 'eff': eff_pre, 'mob': mob_pre, 'res': res_pre}
ARCH_DISPLAY = {'custom':'Custom CNN','eff':'EfficientNetB0','mob':'MobileNetV2','res':'ResNet50'}
SOURCES      = ['aptos', 'eyepacs', 'messidor']
PAIRS        = [('aptos','eyepacs'), ('aptos','messidor'), ('eyepacs','messidor')]

SS_PATHS, POOLED_PATHS = {}, {}
for arch in ARCH_SPECS:
    for src in SOURCES:
        p = f'{SS_DIR}ss_{arch}_{src}_dr.keras'
        assert os.path.exists(p), f"MISSING: {p}"
        SS_PATHS[(arch, src)] = p
POOLED_FILENAMES = {'custom':'dr_best_custom_cnn.keras','eff':'dr_best_efficientnet.keras',
                     'mob':'dr_best_mobilenet.keras','res':'dr_best_resnet50.keras'}
for arch, fn in POOLED_FILENAMES.items():
    p = WEIGHTS_DIR + fn
    assert os.path.exists(p), f"MISSING: {p}"
    POOLED_PATHS[arch] = p
print("All 12 single-source + 4 pooled checkpoints found.")

def make_eval_gen(df, preprocess_fn):
    idg = (ImageDataGenerator(rescale=1./255) if preprocess_fn is None
           else ImageDataGenerator(preprocessing_function=preprocess_fn))
    return idg.flow_from_dataframe(df, x_col='image_path', y_col='grade',
                                    target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                                    class_mode='categorical', classes=GRADES,
                                    color_mode='rgb', shuffle=False)

def extract_penultimate_features(model, df, preprocess_fn):
    # layer[-3] is the 256-unit Dense(relu) output, identical index across all four
    # architectures given the shared head structure (..., GAP, Dense(256,relu), Dropout, Dense(5,softmax))
    feat_model = Model(inputs=model.input, outputs=model.layers[-3].output)
    gen = make_eval_gen(df, preprocess_fn)
    feats = feat_model.predict(gen, verbose=0)
    del feat_model
    return feats

# ================================================================
# EXTRACTION, ONCE. Every gamma convention below reuses these cached features.
# ================================================================
features_ss, features_pooled = {}, {}

print("\n===== EXTRACTING FEATURES, SINGLE-SOURCE MODELS (36 passes: 12 models x 3 test sources) =====")
for arch, preproc in ARCH_SPECS.items():
    for train_src in SOURCES:
        m = load_model(SS_PATHS[(arch, train_src)])
        for test_src in SOURCES:
            feats = extract_penultimate_features(m, test_sets[test_src], preproc)
            features_ss[(arch, train_src, test_src)] = feats
        print(f"  {ARCH_DISPLAY[arch]:15} trained-on-{train_src:9} -> extracted on all 3 sources")
        del m; gc.collect(); tf.keras.backend.clear_session()

print("\n===== EXTRACTING FEATURES, POOLED MODELS (12 passes: 4 models x 3 test sources) =====")
for arch, preproc in ARCH_SPECS.items():
    m = load_model(POOLED_PATHS[arch])
    for test_src in SOURCES:
        feats = extract_penultimate_features(m, test_sets[test_src], preproc)
        features_pooled[(arch, test_src)] = feats
    print(f"  {ARCH_DISPLAY[arch]:15} pooled -> extracted on all 3 sources")
    del m; gc.collect(); tf.keras.backend.clear_session()

np.savez('/kaggle/working/dr_stage13_raw_features.npz',
         **{f"ss_{a}_{tr}_{te}": v for (a,tr,te), v in features_ss.items()},
         **{f"pooled_{a}_{te}": v for (a,te), v in features_pooled.items()})
print("\nSaved dr_stage13_raw_features.npz (raw 256-d features, re-derive any gamma convention from this without re-extracting)")

# ================================================================
# MMD MATH
# ================================================================
def rbf_kernel_mean(A, B, gamma, chunk=500):
    total, count = 0.0, 0
    for i in range(0, A.shape[0], chunk):
        Ai = A[i:i+chunk]
        sq = np.sum((Ai[:, None, :] - B[None, :, :]) ** 2, axis=-1)
        total += np.exp(-gamma * sq).sum()
        count += Ai.shape[0] * B.shape[0]
    return total / count

def rbf_mmd2(X, Y, gamma):
    return (rbf_kernel_mean(X, X, gamma) + rbf_kernel_mean(Y, Y, gamma)
            - 2 * rbf_kernel_mean(X, Y, gamma))

def median_heuristic_gamma(feature_arrays, sample_cap=2000, seed=SEED):
    pooled = np.vstack(feature_arrays)
    rng = np.random.RandomState(seed)
    if len(pooled) > sample_cap:
        pooled = pooled[rng.choice(len(pooled), size=sample_cap, replace=False)]
    sq_all = []
    for i in range(0, len(pooled), 500):
        Ai = pooled[i:i+500]
        sq = np.sum((Ai[:, None, :] - pooled[None, :, :]) ** 2, axis=-1)
        sq_all.append(sq[sq > 0])
    med = np.median(np.concatenate(sq_all))
    return 1.0 / (2.0 * med) if med > 0 else 1.0

# ---------- THREE GAMMA CONVENTIONS ----------
all_feats = list(features_ss.values()) + list(features_pooled.values())
GAMMA_GLOBAL = median_heuristic_gamma(all_feats)

GAMMA_PER_ARCH = {}
for arch in ARCH_SPECS:
    arch_feats = ([v for (a,_,_),v in features_ss.items() if a==arch] +
                  [v for (a,_),v in features_pooled.items() if a==arch])
    GAMMA_PER_ARCH[arch] = median_heuristic_gamma(arch_feats)

GAMMA_FIXED = 1.0 / 256  # standard 1/n_features heuristic, chosen in advance, not derived from this data

print(f"\nGamma, global (one value, everything):        {GAMMA_GLOBAL:.6f}")
print(f"Gamma, per-architecture:                        " +
      ", ".join(f"{ARCH_DISPLAY[a]}={g:.6f}" for a,g in GAMMA_PER_ARCH.items()))
print(f"Gamma, fixed (1/256, not data-derived):          {GAMMA_FIXED:.6f}")

# ================================================================
# STAGE 12 REFERENCE (for mmd_vs_drop files), hardcoded from the locked Stage 12 run
# ================================================================
STAGE12_ROWS = [
 {'arch':'custom','train_source':'aptos','test_source':'aptos','macro_auc':0.8427,'qwk':0.6600},
 {'arch':'custom','train_source':'aptos','test_source':'eyepacs','macro_auc':0.4795,'qwk':0.0764},
 {'arch':'custom','train_source':'aptos','test_source':'messidor','macro_auc':0.5406,'qwk':0.2134},
 {'arch':'custom','train_source':'eyepacs','test_source':'aptos','macro_auc':0.4468,'qwk':0.0000},
 {'arch':'custom','train_source':'eyepacs','test_source':'eyepacs','macro_auc':0.5816,'qwk':0.0000},
 {'arch':'custom','train_source':'eyepacs','test_source':'messidor','macro_auc':0.4461,'qwk':0.0000},
 {'arch':'custom','train_source':'messidor','test_source':'aptos','macro_auc':0.4645,'qwk':0.0000},
 {'arch':'custom','train_source':'messidor','test_source':'eyepacs','macro_auc':0.4693,'qwk':0.0000},
 {'arch':'custom','train_source':'messidor','test_source':'messidor','macro_auc':0.6390,'qwk':0.0000},
 {'arch':'eff','train_source':'aptos','test_source':'aptos','macro_auc':0.8883,'qwk':0.7456},
 {'arch':'eff','train_source':'aptos','test_source':'eyepacs','macro_auc':0.6562,'qwk':0.3047},
 {'arch':'eff','train_source':'aptos','test_source':'messidor','macro_auc':0.7361,'qwk':0.2725},
 {'arch':'eff','train_source':'eyepacs','test_source':'aptos','macro_auc':0.6295,'qwk':0.4501},
 {'arch':'eff','train_source':'eyepacs','test_source':'eyepacs','macro_auc':0.6871,'qwk':0.3118},
 {'arch':'eff','train_source':'eyepacs','test_source':'messidor','macro_auc':0.7295,'qwk':0.4017},
 {'arch':'eff','train_source':'messidor','test_source':'aptos','macro_auc':0.5972,'qwk':0.3617},
 {'arch':'eff','train_source':'messidor','test_source':'eyepacs','macro_auc':0.7168,'qwk':0.3951},
 {'arch':'eff','train_source':'messidor','test_source':'messidor','macro_auc':0.8163,'qwk':0.6621},
 {'arch':'mob','train_source':'aptos','test_source':'aptos','macro_auc':0.9066,'qwk':0.7172},
 {'arch':'mob','train_source':'aptos','test_source':'eyepacs','macro_auc':0.6567,'qwk':0.2211},
 {'arch':'mob','train_source':'aptos','test_source':'messidor','macro_auc':0.6919,'qwk':0.2795},
 {'arch':'mob','train_source':'eyepacs','test_source':'aptos','macro_auc':0.6405,'qwk':0.4312},
 {'arch':'mob','train_source':'eyepacs','test_source':'eyepacs','macro_auc':0.6641,'qwk':0.3278},
 {'arch':'mob','train_source':'eyepacs','test_source':'messidor','macro_auc':0.7125,'qwk':0.4727},
 {'arch':'mob','train_source':'messidor','test_source':'aptos','macro_auc':0.6681,'qwk':0.5319},
 {'arch':'mob','train_source':'messidor','test_source':'eyepacs','macro_auc':0.6822,'qwk':0.1143},
 {'arch':'mob','train_source':'messidor','test_source':'messidor','macro_auc':0.8020,'qwk':0.4682},
 {'arch':'res','train_source':'aptos','test_source':'aptos','macro_auc':0.9134,'qwk':0.8481},
 {'arch':'res','train_source':'aptos','test_source':'eyepacs','macro_auc':0.6673,'qwk':0.2251},
 {'arch':'res','train_source':'aptos','test_source':'messidor','macro_auc':0.7791,'qwk':0.3389},
 {'arch':'res','train_source':'eyepacs','test_source':'aptos','macro_auc':0.6431,'qwk':0.5352},
 {'arch':'res','train_source':'eyepacs','test_source':'eyepacs','macro_auc':0.7191,'qwk':0.4016},
 {'arch':'res','train_source':'eyepacs','test_source':'messidor','macro_auc':0.6914,'qwk':0.3903},
 {'arch':'res','train_source':'messidor','test_source':'aptos','macro_auc':0.6275,'qwk':0.4518},
 {'arch':'res','train_source':'messidor','test_source':'eyepacs','macro_auc':0.6907,'qwk':0.3270},
 {'arch':'res','train_source':'messidor','test_source':'messidor','macro_auc':0.8315,'qwk':0.6607},
]
s12_df = pd.DataFrame(STAGE12_ROWS)

# ================================================================
# PER-GAMMA-CONVENTION FILE GENERATION (5 files each)
# ================================================================
def run_gamma_convention(name, gamma_for_arch_fn):
    full_rows = []
    for arch in ARCH_SPECS:
        gamma = gamma_for_arch_fn(arch)
        for src_a, src_b in PAIRS:
            for extraction_src in [src_a, src_b]:
                fa = features_ss[(arch, extraction_src, src_a)]
                fb = features_ss[(arch, extraction_src, src_b)]
                mmd2 = rbf_mmd2(fa, fb, gamma)
                full_rows.append({'model_type':'single_source','arch':ARCH_DISPLAY[arch],
                                   'source_a':src_a,'source_b':src_b,
                                   'extraction_model_source':extraction_src,
                                   'gamma':gamma,'mmd_squared':mmd2,'mmd':np.sqrt(max(mmd2,0))})
        for src_a, src_b in PAIRS:
            fa = features_pooled[(arch, src_a)]
            fb = features_pooled[(arch, src_b)]
            mmd2 = rbf_mmd2(fa, fb, gamma)
            full_rows.append({'model_type':'pooled','arch':ARCH_DISPLAY[arch],
                               'source_a':src_a,'source_b':src_b,
                               'extraction_model_source':'pooled',
                               'gamma':gamma,'mmd_squared':mmd2,'mmd':np.sqrt(max(mmd2,0))})

    full_df = pd.DataFrame(full_rows)
    full_df.to_csv(f'/kaggle/working/dr_stage13_{name}_full.csv', index=False)

    both_dir = full_df[full_df.model_type=='single_source'][
        ['arch','source_a','source_b','extraction_model_source','mmd_squared','mmd']
    ].rename(columns={'extraction_model_source':'through_model_trained_on'})
    both_dir.to_csv(f'/kaggle/working/dr_stage13_{name}_both_directions.csv', index=False)

    pooled_ref = full_df[full_df.model_type=='pooled'].groupby('arch', as_index=False).agg(
        mmd_squared=('mmd_squared','mean'), mmd=('mmd','mean'))
    pooled_ref.insert(1, 'model_type', 'pooled')
    pooled_ref.to_csv(f'/kaggle/working/dr_stage13_{name}_pooled_reference.csv', index=False)

    ss_avg = both_dir.groupby('arch', as_index=False)['mmd'].mean().rename(columns={'mmd':'mmd_single_source_avg'})
    comparison = ss_avg.merge(pooled_ref[['arch','mmd']].rename(columns={'mmd':'mmd_pooled'}), on='arch')
    comparison['reduction'] = comparison['mmd_single_source_avg'] - comparison['mmd_pooled']
    comparison['pct_reduction'] = (comparison['reduction'] / comparison['mmd_single_source_avg'] * 100).round(1)
    comparison.to_csv(f'/kaggle/working/dr_stage13_{name}_pooled_vs_single_source.csv', index=False)

    drop_rows = []
    for _, r in s12_df.iterrows():
        if r.train_source == r.test_source:
            continue
        within = s12_df[(s12_df.arch==r.arch) & (s12_df.train_source==r.train_source) &
                         (s12_df.test_source==r.train_source)].iloc[0]
        auc_drop = within.macro_auc - r.macro_auc
        qwk_drop = within.qwk - r.qwk
        mrow = both_dir[(both_dir.arch==ARCH_DISPLAY[r.arch]) &
                         (both_dir.through_model_trained_on==r.train_source) &
                         (((both_dir.source_a==r.train_source)&(both_dir.source_b==r.test_source)) |
                          ((both_dir.source_a==r.test_source)&(both_dir.source_b==r.train_source)))]
        mmd_val = mrow['mmd'].iloc[0] if len(mrow) else np.nan
        drop_rows.append({'arch':ARCH_DISPLAY[r.arch],'train_source':r.train_source,'test_source':r.test_source,
                           'auc_drop':auc_drop,'qwk_drop':qwk_drop,'mmd':mmd_val})
    drop_df = pd.DataFrame(drop_rows)
    drop_df.to_csv(f'/kaggle/working/dr_stage13_{name}_mmd_vs_drop.csv', index=False)

    r_auc, p_auc = pearsonr(drop_df['mmd'], drop_df['auc_drop'])
    r_qwk, p_qwk = pearsonr(drop_df['mmd'], drop_df['qwk_drop'])
    print(f"\n--- {name} gamma convention ---")
    print(comparison.round(4).to_string(index=False))
    print(f"MMD vs AUC-drop correlation: r={r_auc:.3f} p={p_auc:.3f}")
    print(f"MMD vs QWK-drop correlation: r={r_qwk:.3f} p={p_qwk:.3f}")
    return full_df, both_dir

print("\n" + "="*70 + "\nGAMMA CONVENTION 1: GLOBAL (one value, everything)\n" + "="*70)
full_global, bd_global = run_gamma_convention('global', lambda a: GAMMA_GLOBAL)

print("\n" + "="*70 + "\nGAMMA CONVENTION 2: PER-ARCHITECTURE\n" + "="*70)
full_perarch, bd_perarch = run_gamma_convention('perarch', lambda a: GAMMA_PER_ARCH[a])

print("\n" + "="*70 + "\nGAMMA CONVENTION 3: FIXED (1/256, not data-derived)\n" + "="*70)
full_fixed, bd_fixed = run_gamma_convention('fixed', lambda a: GAMMA_FIXED)

# ================================================================
# CUSTOM CNN vs PRETRAINED, per source pair, across all 3 conventions in one file
# ================================================================
cvp_rows = []
for name, bd in [('global', bd_global), ('perarch', bd_perarch), ('fixed', bd_fixed)]:
    for src_a, src_b in PAIRS:
        sub = bd[(bd.source_a==src_a)&(bd.source_b==src_b)]
        custom_mmd = sub[sub.arch=='Custom CNN']['mmd'].mean()
        pretrained_mmd = sub[sub.arch!='Custom CNN']['mmd'].mean()
        cvp_rows.append({'gamma_convention':name,'source_a':src_a,'source_b':src_b,
                          'custom_cnn_mmd':custom_mmd,'pretrained_avg_mmd':pretrained_mmd,
                          'custom_higher': custom_mmd > pretrained_mmd})
cvp_df = pd.DataFrame(cvp_rows)
cvp_df.to_csv('/kaggle/working/dr_stage13_custom_vs_pretrained_by_pair.csv', index=False)
print("\n===== Custom CNN vs pretrained-average MMD, per pair, all 3 gamma conventions =====")
print(cvp_df.round(4).to_string(index=False))

# ================================================================
# README
# ================================================================
readme = """DR STAGE 13, FILE GUIDE
========================

WHY THREE VERSIONS OF EVERYTHING
MMD needs a "how strict is close" setting (gamma). Comparing MMD values computed with
different gamma settings is invalid, like comparing heights in inches to heights in
centimeters without converting. This stage computes gamma three defensible ways and
reports all three, rather than picking one and hoping the result doesn't depend on it.

  global  : one single gamma value, derived from the data, applied to every comparison
            across all four architectures. Simplest; assumes all architectures' 256-d
            feature spaces are on a comparable natural scale.
  perarch : one gamma per architecture, derived from that architecture's own features
            only, shared across that architecture's single-source and pooled
            comparisons. Acknowledges architectures may sit at different natural scales.
  fixed   : gamma = 1/256, a standard default, decided in advance, not derived from
            this data at all. Removes any risk of the gamma itself being influenced by
            which images happened to be sampled.

FILES, PER GAMMA CONVENTION (5 each, {global,perarch,fixed} x these 5):
  dr_stage13_{name}_full.csv
      Every single MMD comparison computed, single-source and pooled, tagged with
      which gamma convention and value was used. The master file.
  dr_stage13_{name}_both_directions.csv
      Single-source models only. 3 source pairs x 2 directions x 4 architectures = 24 rows.
      "Direction" means which model's learned feature space the comparison is measured
      through (e.g. MMD(aptos,eyepacs) as seen by the aptos-trained model vs as seen
      by the eyepacs-trained model). These can differ because the two models learned
      different mappings from pixels to the 256-d space.
  dr_stage13_{name}_pooled_reference.csv
      One row per architecture: MMD as measured through the pooled (Stage 4-7) model,
      averaged across the 3 source pairs.
  dr_stage13_{name}_pooled_vs_single_source.csv
      The core Stage 13 question: does pooled training pull sources closer together
      in feature space? Positive pct_reduction = yes, pooling reduced MMD.
      Negative = pooling increased MMD (as chest's corrected run found for all 4
      architectures).
  dr_stage13_{name}_mmd_vs_drop.csv
      24 rows (4 architectures x 6 cross-source cells). Tests whether feature-space
      MMD predicts the AUC/QWK drop Stage 12 already measured. Correlation stats
      (Pearson r, p) printed to console when this cell runs, not stored as columns.

OTHER FILES
  dr_stage13_raw_features.npz
      The raw 256-d penultimate-layer features for all 12 single-source and 4 pooled
      models, each on all 3 sources' test sets (capped at 500 images/source). Every
      MMD number in every file above was computed from these. Any future gamma
      convention can be derived from this file alone, no re-extraction needed.
  dr_stage13_custom_vs_pretrained_by_pair.csv
      Is Custom CNN's MMD higher than the pretrained average, per source pair, under
      all 3 gamma conventions at once (9 rows). Tests the manual's core claim: does
      the from-scratch model show more separated features than the pretrained models.

PENULTIMATE LAYER
  layer[-3] of the outer Sequential in every architecture, the 256-unit Dense(relu)
  output immediately before Dropout and the final Dense(5, softmax) classification
  layer. Identical index across all four architectures because they share this head
  structure (only the backbone before it differs).
"""
with open('/kaggle/working/dr_stage13_README.txt', 'w') as f:
    f.write(readme)

print("\n\n===== ALL 17 FILES SAVED =====")
print("15 gamma-convention files (5 x 3), plus custom_vs_pretrained_by_pair.csv, plus README.txt")
print("Plus dr_stage13_raw_features.npz (not counted in the 17, but the source of all of them)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 121.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-23 09:27:00.661934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787477220.684563      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787477220.692074      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787477220.710012      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787477220.710033      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787477220.710036      58 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
aptos: using 500 of 550 test images for feature extraction
eyepacs: using 500 of 550 test images for feature extraction
messidor: using 263 of 263 test images for feature extraction
All 12 single-source + 4 pooled checkpoints found.

===== EXTRACTING FEATURES, SINGLE-SOURCE MODELS (36 passes: 12 models x 3 test sources) =====


I0000 00:00:1787477231.273172      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787477231.279221      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 500 validated image filenames belonging to 5 classes.


I0000 00:00:1787477241.650442     134 cuda_dnn.cc:529] Loaded cuDNN version 91002


Found 500 validated image filenames belonging to 5 classes.
Found 263 validated image filenames belonging to 5 classes.
  Custom CNN      trained-on-aptos     -> extracted on all 3 sources
Found 500 validated image filenames belonging to 5 classes.
Found 500 validated image filenames belonging to 5 classes.
Found 263 validated image filenames belonging to 5 classes.
  Custom CNN      trained-on-eyepacs   -> extracted on all 3 sources
Found 500 validated image filenames belonging to 5 classes.
Found 500 validated image filenames belonging to 5 classes.
Found 263 validated image filenames belonging to 5 classes.
  Custom CNN      trained-on-messidor  -> extracted on all 3 sources
Found 500 validated image filenames belonging to 5 classes.
Found 500 validated image filenames belonging to 5 classes.
Found 263 validated image filenames belonging to 5 classes.
  EfficientNetB0  trained-on-aptos     -> extracted on all 3 sources
Found 500 validated image filenames belonging to 5 classes.
Foun